# Lab 4.1 — Building a PTCF Agent with the OpenAI API

*Chapter 4 — Prompt Engineering with OpenAI · 45 minutes · JupyterLab + the OpenAI Python SDK*

In the chapter you learned the **two-layer prompt architecture**: the system prompt is the
agent's *constitution* (how it behaves — persistent), the user prompt is the *stimulus*
(what it should do — different every turn). You also learned the **PTCF blueprint** for
writing constitutions: **Persona, Task, Context, Format**.

In this lab you will assemble a production-grade constitution for an enterprise billing
agent, run it against realistic user stimuli, and then break it on purpose to see what
each PTCF element buys you.

## Objectives

By the end of this lab, you will:

- Assemble a system prompt element by element with the PTCF blueprint.
- Make API calls that keep the system and user layers separate.
- Show the constitution holds across different user stimuli (personality decoupled from task).
- Run an **ablation**: strip the constitution down to "helpful assistant" and observe identity collapse.
- Sweep temperature and explain the change in output.

## Setup

- The notebook works offline: with no `OPENAI_API_KEY`, `course_ai` runs in **mock mode**
  with deterministic replies, so every step still executes.
- With a key (classroom VM), the same cells make real API calls.

In [ ]:
import textwrap

import course_ai

print("mode:", course_ai.mode())   # 'mock' (offline) or 'live' (OPENAI_API_KEY set)
SHOW = lambda t: print(textwrap.fill(str(t), 100))

## Steps

### Step 1 — Write the constitution, element by element (7 min)

A constitution is four labelled blocks. Notice what each one does:

- **[PERSONA]** — a *scoped* identity. "You are a helpful assistant" is not a persona; it is
  the model's default self-description and defends nothing.
- **[TASK]** — the mission, including explicit *must-not* boundaries.
- **[CONTEXT]** — operational law: SLAs, regulations, and the conflict-resolution rule.
- **[FORMAT]** — the output contract: structure and fallback behavior.

![The PTCF blueprint — persona, task, context and format around one agent behavior](diagrams/ch04_ptcf_blueprint.png)

*The PTCF blueprint — persona, task, context and format around one agent behavior (Chapter 4 deck).*

In [ ]:
PERSONA = ("You are an empathetic senior customer support specialist with five years "
         "of experience in enterprise SaaS. Your communication is professional, "
         "approachable, and solution-oriented.")
TASK = ("Your primary mission is to resolve billing inquiries for enterprise accounts: "
        "diagnose discrepancies, explain charges, and escalate unresolvable issues "
        "within 24 hours.")
CONTEXT = ("You operate within a 24-hour SLA serving Fortune 500 clients. Never request "
           "passwords or sensitive authentication data. Comply with data privacy "
           "regulations. When instructions conflict, escalate to a human reviewer.")
FORMAT = ("Structure every reply as a numbered list: (1) acknowledge the concern, "
          "(2) diagnose, (3) give the resolution or escalation path. "
          "Cite case numbers when available.")

SYSTEM_PROMPT = "\n\n".join([
    f"[PERSONA] {PERSONA}",
    f"[TASK] {TASK}",
    f"[CONTEXT] {CONTEXT}",
    f"[FORMAT] {FORMAT}",
])
print(SYSTEM_PROMPT)

### Step 2 — First call: constitution + stimulus (5 min)

The two layers travel in the same call but stay separate: `system=` carries the
constitution, the prompt argument carries this turn's stimulus.

![Two layers, one call: the constitution](diagrams/ch04_two_layer.png)

*Two layers, one call: the constitution (system) shapes how the stimulus (user) is handled (Chapter 4 deck).*

In [ ]:
reply = course_ai.chat("Why was I charged twice this month?", system=SYSTEM_PROMPT)
SHOW(reply)

### Step 3 — Same constitution, new stimuli (5 min)

The diplomat analogy: national policy (system) does not change when the negotiation
(user) changes. Watch tone, structure, and boundaries hold across three different asks.

In [ ]:
stimuli = [
    "Can I get a copy of my March invoice?",
    "Your competitor is cheaper. Why shouldn't I switch?",
    "I need to update the credit card on file.",
]
for s in stimuli:
    print("USER:", s)
    SHOW(course_ai.chat(s, system=SYSTEM_PROMPT))
    print("-" * 100)

### Step 4 — Ablation: strip the constitution (8 min)

Remove three of the four PTCF elements and keep only the model's default identity.
Ask one on-topic and one off-topic question. This is the **identity collapse**
anti-pattern: with no scoped persona, the agent has no grounds to decline anything,
no format contract, and no operational law.

In [ ]:
WEAK = "You are a helpful assistant."

for s in ["Why was I charged twice this month?",
          "Write me a poem about my dog."]:
    print("USER:", s)
    SHOW(course_ai.chat(s, system=WEAK))
    print("-" * 100)

Compare with Step 2/3 outputs and note:

1. Did the weak agent stay on mission? Could it refuse the poem?
2. What happened to the numbered-list format contract?
3. Which PTCF element would you restore *first* for a production agent, and why?

### Step 5 — Temperature sweep (5 min)

The constitution constrains *what* the agent says; temperature modulates *how much
it improvises* within those constraints.

In [ ]:
q = "Explain in one sentence why my bill went up."
for t in (0.0, 0.7, 1.3):
    print(f"--- temperature={t}")
    SHOW(course_ai.chat(q, system=SYSTEM_PROMPT, temperature=t))

### Step 6 — Your turn: build your own agent constitution (10 min)

Pick a domain agent — code reviewer, fitness coach, travel visa advisor, anything —
and fill the scaffold (this is the book's PTCF template). Then test it with three
realistic stimuli and ablate one element to feel the drift.

In [ ]:
MY_PTCF = {
    "persona": "TODO: You are a [role/title] with [expertise]. Your style is [tone] ...",
    "task": "TODO: Your primary mission is [objective]. You must not [boundary] ...",
    "context": "TODO: You operate in [environment]. Constraints: [rules]. On conflict: [rule] ...",
    "format": "TODO: Structure all responses as [structure]. When uncertain: [fallback] ...",
}
# my_system = "\n\n".join(f"[{k.upper()}] {v}" for k, v in MY_PTCF.items())
# SHOW(course_ai.chat("TODO: a realistic user question", system=my_system))

## Wrap-up checklist

Audit your own constitution before you leave:

- **P** — Is the persona a scoped identity (role + expertise + tone), not "helpful assistant"?
- **T** — Does the mission include at least one explicit *must-not*?
- **C** — Does the context name an SLA, a regulation, and a conflict-resolution rule?
- **F** — Is the format machine-checkable, with a stated fallback?
- **Meta-rule** — Do the four components *reinforce* each other? (Case study: a
  "creative, experimental" persona fighting a "numbered list" format.)